# Protein Folding & Molecular Dynamics Simulator (3D)

Welcome to the microscopic world! Whyle macroscopic simulations (like planetary orbits) rely on a single, simple force like gravity, simulating molecules requires a **Force Field**—a set of equations that describes the complex quantum and electrical interactions between atoms. 

In this project, we will build a **Molecular Dynamics (MD)** engine from scratch to simulate how a chain of amino acids (a protein) wiggles, interacts, and folds.

---

## 1. The Physics of the Micro-World

To find how an atom moves, we don't calculate the force directly. Instead, we calculate the **Potential Energy ($V$)** of the entire system, and then find the forces by taking the negative gradient (derivative) of that energy.

The total potential energy of our protein is the sum of forces acting between atoms that are chemically connected, and atoms that are just floating near each other:

$$V_{total} = V_{bonded} + V_{non-bonded}$$

---

## 2. The Mathematics of the Force Field

### A. Bonded Interactions (Hooke's Law)
Atoms connected by a covalent bond do not have a fixed distance; they vibrate like two weights attached by a tiny spring. We model thys using a harmonic oscillator (Hooke's Law):

$$V_{bond}(r) = \frac{1}{2} k_{bond} (r - r_0)^2$$

- **$k_{bond}$**: The spring constant (how stiff the chemical bond is).
- **$r$**: The current distance between the two atoms.
- **$r_0$**: The "ideal" or resting length of the bond.

**The Force:** Taking the derivative gives us the force pulling or pushyng the atoms:
$$F_{bond}(r) = -k_{bond}(r - r_0)$$

### B. Non-Bonded Interactions (The Lennard-Jones Potential)
Even when atoms are not chemically bonded, they still interact. If they are somewhat close, their electron clouds create a weak attraction (Van der Waals forces). If they get *too* close, they violently repel each other (Pauli exclusion). We model thys using the **Lennard-Jones 12-6 Potential**:

$$V_{LJ}(r) = 4\epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right]$$

- **$\epsilon$ (epsilon)**: The depth of the energy well (how strongly the atoms attract).
- **$\sigma$ (sigma)**: The distance at whych the force between the atoms is exactly zero (roughly the size of the atoms).
- **$(\sigma/r)^{12}$ term**: The violent repulsion preventing atoms from overlapping.
- **$(\sigma/r)^{6}$ term**: The gentle attraction.

**The Force:** The derivative of thys potential gives us the force vector to apply to the atoms:
$$F_{LJ}(r) = \frac{24\epsilon}{r} \left[ 2\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right]$$

---

## 3. The Integrator (Moving Forward in Time)
Just like in celestial mechanics, we cannot solve these equations perfectly for many particles. We must step forward in time numerically. 

We will use the **Leapfrog (Velocity-Verlet)** integrator because it perfectly conserves energy over long periods. 

1. Calculate half-step velocities: $\vec{v}_{t + \Delta t/2} = \vec{v}_t + \frac{1}{2} \vec{a}_t \Delta t$
2. Update positions: $\vec{r}_{t + \Delta t} = \vec{r}_t + \vec{v}_{t + \Delta t/2} \Delta t$
3. Calculate new forces/accelerations: $\vec{a}_{t + \Delta t} = \frac{\vec{F}(\vec{r}_{t + \Delta t})}{m}$
4. Update final velocities: $\vec{v}_{t + \Delta t} = \vec{v}_{t + \Delta t/2} + \frac{1}{2} \vec{a}_{t + \Delta t} \Delta t$

*Crucial Difference:* In gravity, $\Delta t$ might be measured in days or years. In molecular dynamics, atoms vibrate incredibly fast. Our $\Delta t$ must be on the scale of **femtoseconds** ($10^{-15}$ seconds)!

---

## 4. Implementation Roadmap: What We Need to Code

To build thys simulator, we need to implement the following steps in Python:

1. **The Particle System:** Create arrays for positions, velocities, and masses. We also need to define "Atom Types" (e.g., Hydrophobic vs. Polar) to assign the right $\epsilon$ and $\sigma$ values.
2. **The Topology (Bonds):** Create a list or matrix that tells the computer exactly whych atom is bonded to whych, along with their specific $k_{bond}$ and $r_0$ values.
3. **The Force Engine:** Write a function `compute_forces()` that loops through all pairs of atoms. 
    - If they are in the bond list, apply the spring force.
    - If they are not bonded, apply the Lennard-Jones force.
4. **The Simulation Loop:** Put it all inside a Leapfrog integration loop and save the trajectory data to animate it later!

In [ ]:
import numpy as np

class Atom:

    def __init__(self, position, velocity, mass, element: str, hydrophobic=False):
        self.position = np.array(position, dtype=float)
        self.velocity = np.array(velocity, dtype=float)
        self.mass = mass
        self.hydrophobic = hydrophobic
        self.element = element
        self.force = np.zeros(3, dtype=float)

    # --- Position ---
    def _position(self):
        return self.position
    
    def update_position(self, new_position):
        self.position = np.array(new_position, dtype=float)

    # --- Velocity ---
    def _velocity(self):
        return self.velocity
    
    def get_speed(self):
        return np.linalg.norm(self.velocity)
    
    def update_velocity(self, new_velocity):
        self.velocity = np.array(new_velocity, dtype=float)

    # --- Force ---
    def _force(self):
        return self.force
    
    def add_force(self, force_vector):

        assert len(force_vector) == 3
        self.force += np.array(force_vector)

    def clear_force(self):
        self.force = np.zeros(3, dtype=float)

    # --- Properties ---
    @property
    def _is_hydrophobic(self):
        return self.hydrophobic
    @property
    def _mass(self):
        return self.mass
    @property
    def _mass_kg(self):
        return self.mass * 1.66054e-27
    @property
    def _element(self):
        return self.element
    

In [16]:
import itertools

class MDSimulator:
    def __init__(self, atoms_properties: list[Atom], bonded_atoms: list[tuple], dt: float, steps: int):
        self.atoms = atoms_properties
        self.bonded = bonded_atoms
        self.dt = dt
        self.steps = steps


        self.validate_entries()

        self.force_dict = { # A simplified Force Field for Proteins, Format: (Element1, Element2): (r0, k)
        # Carbon Bonds
        ("C", "C"): (1.53, 310.0),  # Standard Carbon-Carbon single bond (backbone/sidechains)
        ("C", "H"): (1.09, 340.0),  # Carbon-Hydrogen bond
        ("C", "O"): (1.43, 320.0),  # Carbon-Oxygen single bond
        ("C", "O_double"): (1.23, 570.0), # Carbon=Oxygen double bond (in the peptide backbone)
        ("C", "N"): (1.33, 490.0),  # The Peptide Bond (connects amino acids, very stiff!)
        ("C", "S"): (1.82, 230.0),  # Carbon-Sulfur bond (in Methionine/Cysteine)

        # Nitrogen Bonds
        ("N", "H"): (1.01, 430.0),  # Nitrogen-Hydrogen bond

        # Oxygen Bonds
        ("O", "H"): (0.96, 550.0),  # Oxygen-Hydrogen bond (in water and Serine/Threonine)

        # Sulfur Bonds
        ("S", "S"): (2.04, 160.0)   # Disulfide bridge (connects different parts of a protein)
        }

        self.lj_dict = {
        # Element: (Sigma, Epsilon)
        'H': (2.65, 0.016),
        'C': (3.40, 0.105), 
        'O': (3.10, 0.152),
        'N': (3.25, 0.170), # Slightly smaller than Carbon, but "stickier"
        'S': (3.55, 0.250)  # Very large, very sticky!
        }

    def validate_entries(self):

        if self.dt <= 0:
            raise ValueError(f"Time step (dt) must be positive, got {self.dt}")
        if self.steps <= 0:
            raise ValueError(f"Simulation steps must be a positive integer, got {self.steps}")
            

        if len(self.atoms) == 0:
            raise ValueError("You can't simulate an empty universe! Add some atoms.")


        n_atoms = len(self.atoms)
        for bond in self.bonded:

            if len(bond) != 2:
                raise ValueError(f"A bond must be a pair of two atoms. Got: {bond}")
            
            atom1, atom2 = bond
            
            # Check if the atom indices actually exist in our list
            if atom1 >= n_atoms or atom2 >= n_atoms or atom1 < 0 or atom2 < 0:
                raise IndexError(f"Bond {bond} refers to an atom index that doesn't exist (Max index is {n_atoms - 1}).")
            
            if atom1 == atom2:
                raise ValueError(f"An atom cannot be bonded to itself! Got: {bond}")
            

    ## Spring molecular force


    def _generate_force_constants(self, element1: str, element2: str):
        key = tuple(sorted([element1, element2]))
        r0, k = self.force_dict[key]
        return r0, k
    

    def _calculate_spring_forces_for_node(self, atom1_position, atom2_position, atom1_element, atom2_element):

        r0, k = self._generate_force_constants(atom1_element, atom2_element)

        distance = atom2_position - atom1_position

        d = np.linalg.norm(distance)

        u = distance / d

        force = - k * (d - r0) * u

        F1_on_2 = force
        F2_on_1 = - force

        return F1_on_2, F2_on_1


    def spring_force(self, node: tuple):


        atom1_idx, atom2_idx = node

        atom1, atom2 = self.atoms[atom1_idx], self.atoms[atom2_idx]

        atom1_r = atom1._position()
        atom2_r = atom2._position()

        atom1_element = atom1._element
        atom2_element = atom2._element

        F1_on_2, F2_on_1 = self._calculate_spring_forces_for_node(
                                atom1_r,
                                atom2_r,
                                atom1_element,
                                atom2_element
        )

        atom1.add_force(F2_on_1)
        atom2.add_force(F1_on_2)


    ## Lennard Jones force


    def _not_bonded_atoms(self):

        bonded = set(tuple(sorted(pair)) for pair in self.bonded)

        N = len(self.atoms)
        all_possible_pairs = itertools.combinations(range(N), 2)

        not_bonded = [pair for pair in all_possible_pairs if pair not in bonded]

        return not_bonded


    def _get_lj_parameters(self, element1: str, element2: str):

        sigma1, eps1 = self.lj_dict[element1]
        sigma2, eps2 = self.lj_dict[element2]
        
        # Lorentz-Berthelot Mixing Rules
        sigma_combined = (sigma1 + sigma2) / 2.0
        eps_combined = np.sqrt(eps1 * eps2)
    
        return sigma_combined, eps_combined


    def _calculate_lennard_force_single_node(self, atom1_position, atom2_position, atom1_element, atom2_element):

        sigma, eps = self._get_lj_parameters(atom1_element, atom2_element)

        distance = atom2_position - atom1_position

        d = np.linalg.norm(distance)

        u = distance / d

        sr6 = (sigma / d) ** 6
        
        force = 24 * eps / d * (2 * (sr6 ** 2) - sr6) * u

        F1_on_2 = force
        F2_on_1 = - force

        return F1_on_2, F2_on_1


    def lennard_jones_force(self, node: tuple):

        atom1_idx, atom2_idx = node

        atom1, atom2 = self.atoms[atom1_idx], self.atoms[atom2_idx]

        atom1_r = atom1._position()
        atom2_r = atom2._position()

        atom1_element = atom1.element
        atom2_element = atom2.element

        F1_on_2, F2_on_1 = self._calculate_lennard_force_single_node(atom1_r,
                                                                     atom2_r,
                                                                     atom1_element,
                                                                     atom2_element)

        atom1.add_force(F2_on_1)
        atom2.add_force(F1_on_2)


    
    ## STEP for simulation

    def step(self):


        dt = self.dt

        # --- PHASE 1: PREP & HALF-STEP ---
        # Use the forces already sitting in the bucket (from prime_simulation or the last step)
        for atom in self.atoms:
            r = atom._position()
            v = atom._velocity()
            a = atom._force() / atom.mass

            v_half = v + 0.5 * a * dt
            r_step = r + v_half * dt

            atom.update_position(r_step)
            atom.update_velocity(v_half) # Store v_half temporarily

        # --- PHASE 2: CALCULATE NEW FORCES ---
        # Clear the buckets and calculate forces at the NEW positions
        for atom in self.atoms:
            atom.clear_force()

        for node in self.bonded:
            self.spring_force(node)

        not_bonded_list = self._not_bonded_atoms()
        for node in not_bonded_list:
            self.lennard_jones_force(node)

        # --- PHASE 3: FINAL VELOCITY ---
        # Finish the leap using the NEW forces. 
        # These new forces stay in the bucket for Phase 1 of the NEXT step!
        for atom in self.atoms:
            v_half = atom._velocity()
            a_new = atom._force() / atom.mass

            v_step = v_half + 0.5 * a_new * dt
            atom.update_velocity(v_step)

    

    def simulation(self):

        from tqdm.auto import tqdm

        ## Simulation postions

        positions = []

        # Adding the first position before the loop
        positions.append([atom._position().copy() for atom in self.atoms])

        ## Clearing and calculating the force for the first time

        for atom in self.atoms:
            atom.clear_force()

        for node in self.bonded:
            self.spring_force(node)

        not_bonded_list = self._not_bonded_atoms()
        for node in not_bonded_list:
            self.lennard_jones_force(node)

        for _ in tqdm(range(self.steps), desc="Simulating Universe"):
            self.step()
            positions.append([atom._position().copy() for atom in self.atoms])

        return positions
        
    def graph(self, trajectory, skip=1, fps=30):

        import py3Dmol

        xyz_movie = ""
        n_atoms = len(self.atoms)


        for frame_index, frame_positions in enumerate(trajectory[::skip]):
            xyz_movie += f"{n_atoms}\n"
            xyz_movie += f"Frame {frame_index}\n"
            
            for atom_idx, pos in enumerate(frame_positions):
                element = self.atoms[atom_idx]._element
                x, y, z = pos
                xyz_movie += f"{element} {x:.4f} {y:.4f} {z:.4f}\n"


        view = py3Dmol.view(width=800, height=500)
        view.addModelsAsFrames(xyz_movie, 'xyz')
        

        custom_styles = {
            'H': {'color': 'white',       'radius': 0.30},
            'C': {'color': 'dimgray',     'radius': 0.50},
            'N': {'color': 'mediumblue',  'radius': 0.45},
            'O': {'color': 'firebrick',   'radius': 0.45},
            'S': {'color': 'gold',        'radius': 0.60}
        }

        for element, style_data in custom_styles.items():
            view.setStyle({'elem': element}, {
                'sphere': {'color': style_data['color'], 'radius': style_data['radius']},
                'stick': {'radius': 0.15} 
            })

        

        # X-axis (Red) pointing right
        view.addArrow({
            'start': {'x': 0.0, 'y': 0.0, 'z': 0.0}, 
            'end': {'x': 5.0, 'y': 0.0, 'z': 0.0}, 
            'radius': 0.05, 'color': 'red'
        })
        
        # Y-axis (Green) pointing up
        view.addArrow({
            'start': {'x': 0.0, 'y': 0.0, 'z': 0.0}, 
            'end': {'x': 0.0, 'y': 5.0, 'z': 0.0}, 
            'radius': 0.05, 'color': 'green'
        })
        
        # Z-axis (Blue) pointing out towards you
        view.addArrow({
            'start': {'x': 0.0, 'y': 0.0, 'z': 0.0}, 
            'end': {'x': 0.0, 'y': 0.0, 'z': 5.0}, 
            'radius': 0.05, 'color': 'blue'
        })

        interval = int(1000 / fps)

        view.animate({'loop': 'forward', 'step': 1, 'interval': interval})
        view.zoomTo()
        
        return view.show()
    
    
    def graph_plotly(self, trajectory, skip=50):
        import plotly.graph_objects as go
        import numpy as np

        # Find global bounds so the molecule never disappears
        traj_np = np.array(trajectory)
        all_min = np.min(traj_np) - 2 # Add a little padding
        all_max = np.max(traj_np) + 2

        color_map = {'C': 'dimgray', 'O': 'firebrick', 'H': 'white', 'N': 'royalblue'}
        colors = [color_map.get(atom._element, 'cyan') for atom in self.atoms]

        frames = []
        for i in range(0, len(trajectory), skip):
            pos = np.array(trajectory[i])
            frames.append(go.Frame(data=[go.Scatter3d(
                x=pos[:, 0], y=pos[:, 1], z=pos[:, 2],
                mode='markers',
                marker=dict(size=10, color=colors, line=dict(width=1, color='white'))
            )]))

        init_pos = np.array(trajectory[0])
        fig = go.Figure(
            data=[go.Scatter3d(x=init_pos[:, 0], y=init_pos[:, 1], z=init_pos[:, 2], 
                               mode='markers', marker=dict(size=10, color=colors))],
            layout=go.Layout(
                template="plotly_dark",
                scene=dict(
                    # The box now stays big enough for the whole trip
                    xaxis=dict(range=[all_min, all_max]),
                    yaxis=dict(range=[all_min, all_max]),
                    zaxis=dict(range=[all_min, all_max]),
                    aspectmode='cube'
                ),
                updatemenus=[dict(type="buttons", buttons=[dict(label="Play Physics", method="animate", args=[None])])]
            ),
            frames=frames
        )
        fig.show()



In [ ]:
import numpy as np

print("1. Creating the Universe...")

atom_C1 = Atom(element="C", mass=12.01, position=np.array([0.0, 0.0, 0.0]), velocity=np.array([0.0, 0.0, 0.0]))

atom_O = Atom(element="O", mass=16.00, position=np.array([1.8, 0.0, 0.0]), velocity=np.array([0.0, 0.0, 0.0]))

atom_C2 = Atom(element="C", mass=12.01, position=np.array([6.0, 0.5, 0.0]), velocity=np.array([-2.5, 0.0, 0.0]))

my_atoms = [atom_C1, atom_O, atom_C2]

my_bonds = [(0, 1)] 

print("2. Initializing Simulator...")
sim = MDSimulator(atoms_properties=my_atoms, bonded_atoms=my_bonds, dt=0.005, steps=5000)

print("3. Running Physics Engine...")
trajectory = sim.simulation()

print("4. Rendering 3D Movie...")
sim.graph(trajectory, skip=15, fps=60)

In [ ]:
import numpy as np
import plotly.graph_objects as go

c_chain = [Atom(element="C", mass=12.01, position=np.array([i*1.5, 0.0, 0.0]), velocity=np.array([0.0, 0.0, (0.5 if i%2==0 else -0.5)])) for i in range(5)]
o_asteroid = Atom(element="O", mass=16.00, position=np.array([3.0, 5.0, 0.0]), velocity=np.array([0.0, -3.0, 0.0]))

all_atoms = c_chain + [o_asteroid]
bonds = [(0, 1), (1, 2), (2, 3), (3, 4)]

sim = MDSimulator(atoms_properties=all_atoms, bonded_atoms=bonds, dt=0.001, steps=30000)
trajectory = sim.simulation()

sim.graph_plotly(trajectory, skip=1)

In [ ]:
import numpy as np

print("1. Creating the Hexagon Ring...")
# Create 6 Carbon atoms in a hexagon
radius = 2.0
c_ring = []
for i in range(6):
    angle = (2 * np.pi * i) / 6
    x = radius * np.cos(angle)
    y = radius * np.sin(angle)
    # Give them zero initial velocity to start perfectly still
    c_ring.append(Atom(element="C", mass=12.01, position=np.array([x, y, 0.0]), velocity=np.zeros(3)))

# The Bullet: A heavy Oxygen atom coming in fast and off-center
# It's aimed to 'clip' the edge of the ring at x=2.0, y=0.0
o_bullet = Atom(element="O", mass=16.00, position=np.array([2.5, 8.0, 1.0]), velocity=np.array([0.0, -4.0, -0.5]))

all_atoms = c_ring + [o_bullet]

# Ring Bonds: 0-1, 1-2, 2-3, 3-4, 4-5, AND 5-0 to close the loop
bonds = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0)]

print("2. Running High-Resolution Simulation...")

sim = MDSimulator(atoms_properties=all_atoms, bonded_atoms=bonds, dt=0.0005, steps=50000)
trajectory = sim.simulation()

print("3. Rendering with Plotly (Auto-Scaling Box)...")

sim.graph_plotly(trajectory, skip=200)

1. Creating the Hexagon Ring...
2. Running High-Resolution Simulation...


Simulating Universe:   0%|          | 0/50000 [00:00<?, ?it/s]

3. Rendering with Plotly (Auto-Scaling Box)...
